In [62]:
from pathlib import Path
import sys

cwd = Path.cwd()
project_root = next(
    (folder for folder in (cwd, *cwd.parents)
     if (folder / "scripts" / "__init__.py").is_file()),
    None,
)

if project_root is None:
    raise FileNotFoundError(f"Couldn't find the project's scripts folder from {cwd}")

sys.path.insert(0, str(project_root))

from scripts.listing_summarizer import ListingSummarizer
from scripts.answerability_checker import AnswerabilityChecker, search_listings

summarizer = ListingSummarizer()
print("Week 8 imports loaded.")

Week 8 imports loaded.


In [63]:
import json

signals_path = project_root / "processed" / "week6_listing_signals.json"

with signals_path.open(encoding="utf-8") as file:
    signals = json.load(file)

if isinstance(signals, list):
    print(f"Loaded {len(signals):,} records")
    if signals:
        print(signals[0])
elif isinstance(signals, dict):
    print("First five top-level keys:", list(signals)[:5])

Loaded 55,212 records
{'listing_id': '1118422731', 'entities': {'bedrooms': None, 'bathrooms': None, 'price': None, 'sqft': None, 'amenities': []}, 'amenities': [], 'condition_keywords': [], 'financing_terms': [], 'location_features': ['cul_de_sac']}


In [64]:
if isinstance(signals, list):
    listing = signals[0]
else:
    listing_id, listing = next(iter(signals.items()))
    print("First key:", listing_id)

if isinstance(listing, dict):
    for key, value in listing.items():
        print(f"{key}: {repr(value)[:250]}")
else:
    print(repr(listing)[:1000])

listing_id: '1118422731'
entities: {'bedrooms': None, 'bathrooms': None, 'price': None, 'sqft': None, 'amenities': []}
amenities: []
condition_keywords: []
financing_terms: []
location_features: ['cul_de_sac']


In [65]:
if isinstance(signals, list):
    listing = signals[0]
else:
    listing_id, listing = next(iter(signals.items()))
    print("First key:", listing_id)

if isinstance(listing, dict):
    for key, value in listing.items():
        print(f"{key}: {repr(value)[:250]}")
else:
    print(repr(listing)[:1000])

listing_id: '1118422731'
entities: {'bedrooms': None, 'bathrooms': None, 'price': None, 'sqft': None, 'amenities': []}
amenities: []
condition_keywords: []
financing_terms: []
location_features: ['cul_de_sac']


In [66]:
# ============================================================
# STEP 3: LOAD REAL ESTATE LISTINGS FROM MYSQL
# ============================================================

# Import libraries for database connections and data handling.
import os
import mysql.connector
import pandas as pd

# Import getpass to securely enter the database password.
# The password will not be displayed when typed.
from getpass import getpass

# Connect to the MySQL database.
# The password is entered privately when the cell runs,
# rather than being saved directly in the notebook.
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=getpass("Enter MySQL password: "),
    database="real_estate"
)

# Retrieve all real estate listings.
# Include structured fields for evaluating extraction accuracy.
# No LIMIT is used because Week 6 requires the entire table.
query = """
SELECT
    L_ListingID,
    L_Address,
    L_City,
    L_Keyword2 AS beds,
    LM_Dec_3 AS baths,
    L_SystemPrice AS price,
    L_Remarks
FROM rets_property
"""

# Load the SQL results into a pandas DataFrame.
try:
    df = pd.read_sql(query, conn)

finally:
    # Close the connection even if loading fails.
    conn.close()

# Display the number of listings loaded.
print(f"Total listings loaded: {len(df):,}")

# Display the first five records.
display(df.head())

# Display the available column names.
print("\nAvailable columns:")
print(df.columns.tolist())

C:\Users\luoxu\AppData\Local\Temp\ipykernel_15284\84944378.py:41: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Total listings loaded: 55,212


,L_ListingID,L_Address,L_City,beds,baths,price,L_Remarks
0,1118422731,1461 Laurel Way,Beverly Hills,4.0,5.0,3950000,Opportunity to reimagine a Classic 70's Archit...
1,1118398412,582 Pasture Ave,Lathrop,4.0,3.0,775000,Step into comfort and functionality with this ...
2,1118383115,14947 Dickens Street 3,Sherman Oaks,2.0,2.0,750000,WOW! NOW REDUCED AND THE SELLER MAY FINANCE! C...
3,1118381441,349 E Gobind Blvd,Mountain House,5.0,4.0,1122426,Plan 3 features an open floor plan with covere...
4,1118372790,515 E Fiorino Street,Mountain House,5.0,3.0,1128292,Plan 2 features an open floor plan with covere...



Available columns:
['L_ListingID', 'L_Address', 'L_City', 'beds', 'baths', 'price', 'L_Remarks']


In [67]:

# Look up the same listing in the Week 6 signals and the MySQL data.
listing_id = "1118422731"

listing_ids = df["L_ListingID"].astype("string").str.strip()
matches = df.loc[listing_ids.eq(listing_id)]

if len(matches) != 1:
    raise ValueError(f"Expected one MySQL row for {listing_id}, found {len(matches)}.")

row = matches.iloc[0]

# Start with the Week 6 features that describe the property.
features = []
for field in ("amenities", "condition_keywords", "location_features"):
    for feature in listing.get(field) or []:
        phrase = feature.replace("_", " ").strip()
        if phrase and phrase not in features:
            features.append(phrase)

entities = {
    "bedrooms": row["beds"],
    "bathrooms": row["baths"],
    "price": row["price"],
    "location": row["L_City"],
    "features": features[:2],
}

print("Listing ID:", listing_id)
print("Remarks:", row["L_Remarks"])
print("Summary:", summarizer.summarize(row["L_Remarks"], entities))

Listing ID: 1118422731
Remarks: Opportunity to reimagine a Classic 70's Architectural property located on a quiet cul-de-sac in BHPO. This home was build in 1973  in the Mid Century era. The breakfast and kitchen areas provide picturesque views of the city down below and as far as the coastline. The primary bedroom suite features a walk in closet plus additional closet space. There are 2 additional bedrooms, a bathroom and powder room adjacent to the Primary Suite. On the entry level there is a bedroom and bathroom. Currently the home is located in the Warner Avenue School District. Broker does not guarantee the accuracy on the information provided, including square footage, lot size, permitted and unpermitted spaces and/or bedroom/bathroom count. Buyer is advised to independently verify the accuracy of this information through personal inspection, building permits and/or appropriate professionals (i.e. attorney, appraiser architect, surveyor, etc.).
Summary: The listing shows 4 beds, 

In [68]:
# Manually verify the two features for this example.
example_entities = {
    **entities,
    "features": ["cul-de-sac", "city and coastline views"],
}

summary = summarizer.summarize(row["L_Remarks"], example_entities)
print(summary)

The listing shows 4 beds, 5 baths, Beverly Hills, listed at $3,950,000. Features include cul-de-sac and city and coastline views.


In [ ]:
def _features_from_listing(self, remarks, entities):
    features = self._features(entities)
    features = [
        "cul-de-sac" if feature.casefold() == "cul de sac" else feature
        for feature in features
    ]

    if isinstance(remarks, str):
        feature_patterns = [
            (r"\bviews? of the city\b[^.!?]*\bcoastline\b",
             "city and coastline views"),
            (r"\bwalk[- ]in closet\b", "walk-in closet"),
        ]

        for pattern, label in feature_patterns:
            if len(features) >= 2:
                break
            if re.search(pattern, remarks, flags=re.IGNORECASE):
                if label.casefold() not in {feature.casefold() for feature in features}:
                    features.append(label)

    return features

In [70]:
import importlib
from scripts import listing_summarizer

# Load the latest saved version of scripts/listing_summarizer.py.
importlib.reload(listing_summarizer)

# Create a summarizer using the updated class.
summarizer = listing_summarizer.ListingSummarizer()

# Use the original MySQL remarks and listing facts from the earlier cell.
# The original entities let us test whether the second feature is found automatically.
remarks = row["L_Remarks"]
summary = summarizer.summarize(remarks, entities)

print("Summary:", summary)

Summary: The listing shows 4 beds, 5 baths, Beverly Hills, listed at $3,950,000. Features include cul-de-sac and city and coastline views.


In [71]:
import importlib
from scripts import listing_summarizer

# Reload the saved Python file and show which copy the notebook uses.
importlib.reload(listing_summarizer)
print("Loaded from:", listing_summarizer.__file__)

summarizer = listing_summarizer.ListingSummarizer()
remarks = row["L_Remarks"]

# Check which features the updated helper finds before building the summary.
if hasattr(summarizer, "_features_from_listing"):
    print("Detected features:", summarizer._features_from_listing(remarks, entities))
else:
    print("The loaded class is missing _features_from_listing.")

print("Summary:", summarizer.summarize(remarks, entities))

Loaded from: c:\Users\luoxu\OneDrive - University of Illinois - Urbana\idx-exchange\IDX-Exchange-NLP\scripts\listing_summarizer.py
Detected features: ['cul-de-sac', 'city and coastline views']
Summary: The listing shows 4 beds, 5 baths, Beverly Hills, listed at $3,950,000. Features include cul-de-sac and city and coastline views.


In [72]:
from pathlib import Path
from scripts import listing_summarizer

# Find the project root from the imported script.
project_root = Path(listing_summarizer.__file__).resolve().parents[1]
reference_path = project_root / "processed" / "week8_reference_summaries.csv"

# Preserve any reference summaries already entered in the CSV.
if reference_path.exists():
    print(f"Using existing reference sheet: {reference_path}")
else:
    reference_sheet.to_csv(reference_path, index=False)
    print(f"Saved {len(reference_sheet)} listings to {reference_path}")

Using existing reference sheet: C:\Users\luoxu\OneDrive - University of Illinois - Urbana\idx-exchange\IDX-Exchange-NLP\processed\week8_reference_summaries.csv


In [73]:
from pathlib import Path
from scripts import listing_summarizer

# The script is in <project root>/scripts/listing_summarizer.py.
project_root = Path(listing_summarizer.__file__).resolve().parents[1]
reference_path = project_root / "processed" / "week8_reference_summaries.csv"

# Save the blank sheet once so rerunning this cell preserves your edits.
if reference_path.exists():
    print(f"File already exists: {reference_path}")
else:
    reference_sheet.to_csv(reference_path, index=False)
    print(f"Saved {len(reference_sheet)} listings to {reference_path}")

File already exists: C:\Users\luoxu\OneDrive - University of Illinois - Urbana\idx-exchange\IDX-Exchange-NLP\processed\week8_reference_summaries.csv


In [74]:
# Read listing IDs as text so they match the IDs in test_df.
references = pd.read_csv(reference_path, dtype={"listing_id": "string"})
references["listing_id"] = references["listing_id"].str.strip()
references["reference_summary"] = references["reference_summary"].fillna("").str.strip()

# Join by listing ID so the CSV rows can be in any order.
test_df = test_df.drop(columns="reference_summary", errors="ignore").merge(
    references[["listing_id", "reference_summary"]],
    on="listing_id",
    how="left",
    validate="one_to_one",
)

missing = test_df["reference_summary"].fillna("").eq("")
if missing.any():
    print("Listings that still need a reference summary:")
    display(test_df.loc[missing, ["listing_id", "remarks"]])
else:
    print(f"Ready for ROUGE-L: {len(test_df)} completed reference summaries.")

Ready for ROUGE-L: 20 completed reference summaries.


In [75]:
import html

import ipywidgets as widgets
import pandas as pd
from IPython.display import display

# Load the existing reference sheet and preserve any summaries already written.
sheet = pd.read_csv(reference_path, dtype={"listing_id": "string"})
sheet["reference_summary"] = sheet["reference_summary"].fillna("")

# Start with the first unfinished listing.
unfinished = sheet.index[sheet["reference_summary"].str.strip().eq("")]
start = int(unfinished[0]) if len(unfinished) else 0

listing_picker = widgets.Dropdown(
    options=[
        (f"{i + 1} of {len(sheet)} — {row['listing_id']}", i)
        for i, row in sheet.iterrows()
    ],
    value=start,
    description="Listing",
    layout=widgets.Layout(width="420px"),
)

listing_details = widgets.HTML()
summary_box = widgets.Textarea(
    placeholder="Write 2–3 sentences using the listing facts and remarks.",
    layout=widgets.Layout(width="100%", height="120px"),
)
save_button = widgets.Button(
    description="Save & next",
    button_style="success",
)
status = widgets.Label()


def show_listing(change=None):
    """Show the selected listing and any saved reference."""
    row = sheet.loc[listing_picker.value]
    remarks = html.escape(str(row["remarks"]))
    city = html.escape(str(row["L_City"]))

    listing_details.value = (
        f"<b>{city}</b> · {row['beds']:g} beds · {row['baths']:g} baths"
        f" · ${row['price']:,.0f}"
        "<p><b>Full listing remarks</b></p>"
        '<div style="white-space: pre-wrap; max-height: 320px; overflow-y: auto;">'
        f"{remarks}</div>"
    )
    summary_box.value = row["reference_summary"]

    completed = sheet["reference_summary"].str.strip().ne("").sum()
    status.value = f"{completed} of {len(sheet)} references saved"


def save_and_next(button):
    """Save the current reference and advance to the next unfinished listing."""
    summary = summary_box.value.strip()
    if not summary:
        status.value = "Enter a reference summary before saving."
        return

    current = listing_picker.value
    sheet.at[current, "reference_summary"] = summary
    sheet.to_csv(reference_path, index=False)

    remaining = sheet.index[sheet["reference_summary"].str.strip().eq("")]
    if len(remaining):
        later = remaining[remaining > current]
        next_listing = int(later[0] if len(later) else remaining[0])
        listing_picker.value = next_listing
    else:
        status.value = f"All {len(sheet)} references saved. Ready for ROUGE-L."


listing_picker.observe(show_listing, names="value")
save_button.on_click(save_and_next)
show_listing()

display(widgets.VBox([
    listing_picker,
    listing_details,
    summary_box,
    save_button,
    status,
]))

In [76]:
import importlib
from scripts import listing_summarizer

# Load the revised class from the saved Python script.
importlib.reload(listing_summarizer)
summarizer = listing_summarizer.RankedListingSummarizer()


class RankedListingSummarizer(ListingSummarizer):
    # Check for specific property details before using broad Week 6 tags.
    FEATURE_PATTERNS = (
        (r"\b(?:bay|ocean|lake|mountain|city|coastline|waterfront) views?\b",
         None),
        (r"\bviews? of the city\b[^.!?]*\bcoastline\b",
         "city and coastline views"),
        (r"\b(?:wheelchair[- ]accessible elevator|lake rights|"
         r"detached RV garage|wraparound deck|covered patio|"
         r"spacious backyard|walk[- ]in closet|"
         r"quartz countertops|solar power|private balcony|"
         r"split[- ]bedroom layout|spacious loft)\b", None),
        (r"\b(?:\d+\s*(?:x|×|by)\s*\d+\s*(?:foot\s+)?shop|workshop)\b", None),
        (r"\bfenced and cross[- ]fenced\b", "fenced and cross-fenced land"),
        (r"\b\d[\d,]*\s+square feet of living space\b", None),
        (r"\b(?:needs updating|ready for some TLC and updating)\b",
         "a layout that needs updating"),
    )

    def _features_from_listing(self, remarks, entities):
        """Rank concrete remarks details ahead of the Week 6 tags."""
        text = remarks if isinstance(remarks, str) else ""
        candidates = []

        for pattern, label in self.FEATURE_PATTERNS:
            match = re.search(pattern, text, flags=re.IGNORECASE)
            if match:
                candidates.append(label or match.group().lower())

        # Keep Week 6 tags as a fallback. A conversion is a possibility.
        garage_conversion = re.search(
            r"\bconvert(?:ed)? back into (?:a )?garage\b", text, re.IGNORECASE
        )
        generic_tags = {"updated", "renovated", "near shopping"}
        for tag in entities.get("features") or []:
            normalized = tag.replace("_", " ").strip().lower()
            if normalized in generic_tags:
                continue
            if normalized == "garage" and garage_conversion:
                continue
            candidates.append(normalized)

        # Reuse the original normalization, deduplication, and two-feature limit.
        return self._features({"features": candidates})


# The scoring cell uses this instance for the revised run.
summarizer = RankedListingSummarizer()

# Preview the same five listings before rerunning ROUGE-L.
for _, listing in evaluation.nsmallest(5, "generated_summary_rouge_l").iterrows():
    print(listing["listing_id"], summarizer.summarize(
        listing["remarks"], listing["entities"]
    ))

1185696125 The listing shows 3 beds, 2 baths, Concord, listed at $670,000. Features include spacious backyard.
1189248361 The listing shows 3 beds, 3 baths, Lake Elsinore, listed at $625,000. Features include spacious loft and 1,868 square feet of living space.
1179270546 The listing shows 2 beds, 3 baths, San Diego, listed at $2,495,000. Features include split-bedroom layout and pool.
1176627746 The listing shows 4 beds, 2 baths, Winchester, listed at $557,000. Features include laundry and garage.
1177020525 The listing shows 2 beds, 2 baths, Joshua Tree, listed at $750,000. Features include parking and deck.


In [77]:
from rouge_score import rouge_scorer

# Read the latest references saved by the widget.
references = pd.read_csv(reference_path, dtype={"listing_id": "string"})
references["listing_id"] = references["listing_id"].str.strip()

evaluation = test_df.drop(columns="reference_summary", errors="ignore").merge(
    references[["listing_id", "reference_summary"]],
    on="listing_id",
    how="left",
    validate="one_to_one",
)

# Check for blank or missing references before scoring.
missing = evaluation["reference_summary"].fillna("").str.strip().eq("")

if missing.any():
    print("Use the widget to finish these listing IDs:")
    print("\n".join(evaluation.loc[missing, "listing_id"]))
else:
    # Compare the first two remarks sentences, pure extraction, and the final summary.
    evaluation["baseline"] = evaluation["remarks"].map(
        lambda remarks: " ".join(summarizer._sentences(remarks)[:2])
    )
    evaluation["extractive"] = [
        summarizer.extractive_summary(remarks, entities)
        for remarks, entities in zip(evaluation["remarks"], evaluation["entities"])
    ]
    evaluation["generated_summary"] = [
        summarizer.summarize(remarks, entities)
        for remarks, entities in zip(evaluation["remarks"], evaluation["entities"])
    ]

    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    mean_scores = {}

    for method in ("baseline", "extractive", "generated_summary"):
        evaluation[f"{method}_rouge_l"] = [
            scorer.score(reference, prediction)["rougeL"].fmeasure
            for reference, prediction in zip(
                evaluation["reference_summary"], evaluation[method]
            )
        ]
        mean_scores[method] = evaluation[f"{method}_rouge_l"].mean()

    display(pd.Series(mean_scores, name="Mean ROUGE-L F1").to_frame())
    print(f"Listings scored: {len(evaluation)}")
    print("ROUGE-L > 0.40:", mean_scores["generated_summary"] > 0.40)

    # Give teammates the source facts, full remarks, and automatic summary.
    review = evaluation[
        ["listing_id", "beds", "baths", "price", "L_City", "remarks", "generated_summary"]
    ].copy()
    for column in ["accuracy_1_to_5", "coverage_1_to_5", "readability_1_to_5", "notes"]:
        review[column] = ""

    # Preserve any teammate ratings if this cell is rerun.
    review_path = reference_path.with_name("week8_teammate_review.csv")
    if review_path.exists():
        print(f"Using existing review sheet: {review_path}")
    else:
        review.to_csv(review_path, index=False)
        print(f"Saved teammate review sheet: {review_path}")

,Mean ROUGE-L F1
baseline,0.240331
extractive,0.258509
generated_summary,0.421857


Listings scored: 20
ROUGE-L > 0.40: True
Using existing review sheet: C:\Users\luoxu\OneDrive - University of Illinois - Urbana\idx-exchange\IDX-Exchange-NLP\processed\week8_teammate_review.csv


In [78]:
# Compare the five lowest-scoring summaries with their references.
lowest = evaluation.nsmallest(5, "generated_summary_rouge_l")

for _, listing in lowest.iterrows():
    print(f"\nListing {listing['listing_id']}")
    print(f"ROUGE-L F1: {listing['generated_summary_rouge_l']:.3f}")
    print(f"Week 6 features: {listing['entities'].get('features', [])}")
    print("Generated:", listing["generated_summary"])
    print("Reference:", listing["reference_summary"])


Listing 1185696125
ROUGE-L F1: 0.262
Week 6 features: ['garage', 'near_shopping']
Generated: The listing shows 3 beds, 2 baths, Concord, listed at $670,000. Features include spacious backyard.
Reference: This single-story, 3-bedroom home sits on a quiet court near downtown Concord and is listed at $670,000. It has 1.5 baths as described in the remarks, a spacious backyard, and a media room that can be converted back into a garage.

Listing 1189248361
ROUGE-L F1: 0.333
Week 6 features: ['patio', 'updated', 'move_in_ready']
Generated: The listing shows 3 beds, 3 baths, Lake Elsinore, listed at $625,000. Features include spacious loft and 1,868 square feet of living space.
Reference: This move-in-ready, 3-bedroom, 3-bath home in Lake Elsinore’s Summerly community is listed at $625,000. It features an open-concept kitchen with granite countertops and a large backyard with a covered patio and no rear neighbors.

Listing 1179270546
ROUGE-L F1: 0.353
Week 6 features: ['pool', 'renovated', 'w

In [79]:
revised_review_path = reference_path.with_name("week8_teammate_review_revised.csv")
if not revised_review_path.exists():
    review.to_csv(revised_review_path, index=False)
print(revised_review_path)

C:\Users\luoxu\OneDrive - University of Illinois - Urbana\idx-exchange\IDX-Exchange-NLP\processed\week8_teammate_review_revised.csv


In [80]:
from scripts.answerability_checker import AnswerabilityChecker, search_listings


# Demo adapters that follow the same interface as the Week 4 components.
class DemoParser:
    def parse(self, query):
        # These three demo queries all ask for Beverly Hills listings.
        return {"city": "Beverly Hills"}


class DemoValidator:
    def validate_query(self, filters):
        return ("city" in filters, [])


checker = AnswerabilityChecker(
    parser=DemoParser(),
    schema_validator=DemoValidator(),
    taxonomy={"hoa": "hoa_fee"},
)


def fetch_city_listings(filters):
    """Search the real listings already loaded from MySQL."""
    city = df["L_City"].astype("string").str.strip().str.casefold()
    matches = city.eq(filters["city"].casefold())
    return df.loc[matches].head(3)


cases = [
    ("Supported search", "Homes in Beverly Hills", fetch_city_listings),
    ("Unsupported criterion", "Homes in Beverly Hills with low HOA", fetch_city_listings),
    ("Zero matches", "Homes in Beverly Hills", lambda filters: df.iloc[:0]),
]

for label, query, fetch in cases:
    result = search_listings(query, checker, fetch)
    print(f"{label}: {result['message']}")
    print(f"Answerable: {result['answerable']} | Results: {len(result['results'])}\n")

Supported search: Results found.
Answerable: True | Results: 3

Unsupported criterion: I couldn't apply the requested criterion: hoa.
Answerable: False | Results: 0

Zero matches: No listings matched your criteria.
Answerable: False | Results: 0



In [82]:
import html
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from IPython.display import display

# Resume an existing self-review or start from the latest scored summaries.
self_review_path = Path(reference_path).with_name("week8_teammate_review_revised.csv")
rating_columns = {
    "accuracy_1_to_5": "Accuracy",
    "coverage_1_to_5": "Coverage",
    "readability_1_to_5": "Readability",
}

if self_review_path.exists():
    self_review = pd.read_csv(self_review_path, dtype={"listing_id": "string"})
else:
    self_review = review.drop(columns=[*rating_columns, "notes"]).copy()
    for column in rating_columns:
        self_review[column] = pd.Series(pd.NA, index=self_review.index, dtype="Int64")
    self_review["notes"] = ""

self_review = self_review.reset_index(drop=True)
self_review["notes"] = self_review["notes"].fillna("")
for column in rating_columns:
    self_review[column] = pd.to_numeric(
        self_review[column], errors="coerce"
    ).astype("Int64")


def unfinished_rows():
    """Find listings that still need at least one rating."""
    return self_review.index[self_review[list(rating_columns)].isna().any(axis=1)]


remaining = unfinished_rows()
picker = widgets.Dropdown(
    options=[
        (f"{i + 1} of {len(self_review)} — {row['listing_id']}", i)
        for i, row in self_review.iterrows()
    ],
    value=int(remaining[0]) if len(remaining) else 0,
    description="Listing",
    layout=widgets.Layout(width="380px"),
)
details = widgets.HTML()
ratings = {
    column: widgets.Dropdown(
        options=[("Select a rating", None), 1, 2, 3, 4, 5],
        description=label,
    )
    for column, label in rating_columns.items()
}
notes = widgets.Textarea(
    description="Notes",
    placeholder="Optional: flag a wrong bath count, missing feature, or awkward wording.",
    layout=widgets.Layout(width="100%", height="80px"),
)
save_button = widgets.Button(description="Save & next", button_style="success")
status = widgets.Label()


def show_listing(change=None):
    """Display the selected listing and restore any saved ratings."""
    row = self_review.loc[picker.value]
    city = html.escape(str(row["L_City"]))
    remarks = html.escape(str(row["remarks"]))
    summary = html.escape(str(row["generated_summary"]))

    details.value = (
        f"<b>{city}</b> · {row['beds']:g} beds · {row['baths']:g} baths"
        f" · ${row['price']:,.0f}"
        f"<p><b>Generated summary</b><br>{summary}</p>"
        "<b>Full listing remarks</b>"
        '<div style="white-space:pre-wrap;max-height:280px;overflow-y:auto;">'
        f"{remarks}</div>"
    )

    for column, control in ratings.items():
        value = row[column]
        control.value = None if pd.isna(value) else int(value)

    notes.value = row["notes"]
    completed = len(self_review) - len(unfinished_rows())
    status.value = f"{completed} of {len(self_review)} summaries rated."


def save_and_next(button):
    """Save the current review and open the next unfinished listing."""
    if any(control.value is None for control in ratings.values()):
        status.value = "Select all three ratings before saving."
        return

    current = picker.value
    for column, control in ratings.items():
        self_review.at[current, column] = control.value
    self_review.at[current, "notes"] = notes.value.strip()
    self_review.to_csv(self_review_path, index=False)

    remaining = unfinished_rows()
    if len(remaining):
        later = remaining[remaining > current]
        picker.value = int(later[0] if len(later) else remaining[0])
    else:
        status.value = f"All {len(self_review)} self-reviews saved."
        display(
            self_review[list(rating_columns)]
            .mean()
            .round(2)
            .rename("Mean rating out of 5")
            .to_frame()
        )


picker.observe(show_listing, names="value")
save_button.on_click(save_and_next)
show_listing()

display(widgets.VBox([
    picker, details, *ratings.values(), notes, save_button, status
]))

,Mean rating out of 5
accuracy_1_to_5,4.70
coverage_1_to_5,4.95
readability_1_to_5,4.95


| Score | Accuracy | Coverage | Readability |
|---|---|---|---|
| **5** | All claims match the source. | Includes all six details: beds, baths, price, location, and two distinct features. | Clear, natural, and concise throughout. |
| **4** | One minor imprecision that doesn’t change the meaning. | Includes five of the six details. | Easy to read, with one slightly awkward phrase. |
| **3** | One substantive error, or several minor imprecisions. | Includes four of the six details. | Understandable, but noticeably choppy, repetitive, or wordy. |
| **2** | Several substantive errors, or one error that materially misrepresents the listing. | Includes two or three of the six details. | Hard to follow in multiple places. |
| **1** | Most claims are incorrect or unsupported. | Includes zero or one of the six details. | Difficult to understand overall. |

## Week 8 Results

The listing summarizer supports extractive sentence selection and produces
short summaries using structured listing facts and features from the remarks.

- **ROUGE-L F1:** The revised summarizer scored **0.4219** across 20
  development listings, up from **0.3792** before feature ranking was revised.
  These listings informed the revision, so held-out performance is unverified.
- **Manual self-review:** I rated 20 revised summaries. Mean ratings were
  **4.70/5 for accuracy, 4.95/5 for coverage, and 4.95/5 for readability**.
  Teammate ratings were unavailable.
- **Answerability:** The notebook demonstrated three outcomes: a supported
  search returned three listings, an unsupported HOA criterion returned an
  explanation, and a simulated empty result returned a no-match message.
  The demo used test adapters for the parser and validator.